In [1]:
from nuscenes.nuscenes import NuScenes

# Initialize the dataset using the relative path you just created
nusc = NuScenes(version='v1.0-mini', dataroot='./data/sets/nuscenes', verbose=True)

# Print out the available scenes to confirm success
nusc.list_scenes()
from nuscenes.utils.data_classes import RadarPointCloud
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
from nuscenes.utils.data_classes import RadarPointCloud, LidarPointCloud
from nuscenes.map_expansion.map_api import NuScenesMap
from pyquaternion import Quaternion


Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 0.421 seconds.
Reverse indexing ...
Done reverse indexing in 0.0 seconds.
scene-0061, Parked truck, construction, intersectio... [18-07-24 03:28:47]   19s, singapore-onenorth, #anns:4622
scene-0103, Many peds right, wait for turning car, ... [18-08-01 19:26:43]   19s, boston-seaport, #anns:2046
scene-0655, Parking lot, parked cars, jaywalker, be... [18-08-27 15:51:32]   20s, boston-seaport, #anns:2332
scene-0553, Wait at intersection, bicycle, large tr... [18-08-28 20:48:16]   20s, boston-seaport, #anns:1950
scene-0757, Arrive at busy intersection, bus, wait ... [18-08-30 19:25:08]   20s, boston-seaport, #anns:592
scene-0796, Scooter, peds on sidewalk, bus, cars, t... [18-10-02 02:52:24]   20s, singapore-queensto, #anns:708
scene-0916, Parki

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
from nuscenes.utils.data_classes import LidarPointCloud, RadarPointCloud
from nuscenes.eval.common.utils import Quaternion
from nuscenes.map_expansion.map_api import NuScenesMap

class NuScenesTemporalDataset(Dataset):
    def __init__(self, nusc_env, scene_list, max_lidar_points=12000, max_radar_points=256, nsweeps=3):
        self.nusc = nusc_env
        self.samples = []
        self.max_lidar_points = max_lidar_points
        self.max_radar_points = max_radar_points # NEW: Enforcing strict radar dimensions
        self.nsweeps = nsweeps
        
        for scene in scene_list:
            current_token = scene['first_sample_token']
            while current_token != '':
                sample = self.nusc.get('sample', current_token)
                self.samples.append(sample)
                current_token = sample['next']
                
        self.img_transform = transforms.Compose([
            transforms.Resize((224, 400)), # Keeping at 224x400 to help your 4GB VRAM
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        locations = ['boston-seaport', 'singapore-onenorth', 'singapore-hollandvillage', 'singapore-queenstown']
        self.nusc_maps = {loc: NuScenesMap(dataroot=self.nusc.dataroot, map_name=loc) for loc in locations}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # 1. Camera Image
        cam_token = sample['data']['CAM_FRONT']
        cam_data = self.nusc.get('sample_data', cam_token)
        image = Image.open(self.nusc.get_sample_data_path(cam_token)).convert('RGB')
        img_tensor = self.img_transform(image)
        
        # 2. RADAR FUSION (Now Padded/Sampled to fixed 256 size)
        radar_pc, radar_times = RadarPointCloud.from_file_multisweep(
            self.nusc, sample, chan='RADAR_FRONT', ref_chan='RADAR_FRONT', nsweeps=self.nsweeps
        )
        radar_base_features = radar_pc.points[[0, 1, 2, 8, 9], :]
        radar_temporal = np.vstack([radar_base_features, radar_times]).T
        
        num_radar = radar_temporal.shape[0]
        if num_radar == 0:
             radar_padded = np.zeros((self.max_radar_points, 6))
        elif num_radar >= self.max_radar_points:
            indices = np.random.choice(num_radar, self.max_radar_points, replace=False)
            radar_padded = radar_temporal[indices, :]
        else:
            indices = np.random.choice(num_radar, self.max_radar_points, replace=True)
            radar_padded = radar_temporal[indices, :]
            
        radar_tensor = torch.tensor(radar_padded, dtype=torch.float32)
        
        # 3. LiDAR FUSION
        lidar_pc, lidar_times = LidarPointCloud.from_file_multisweep(
            self.nusc, sample, chan='LIDAR_TOP', ref_chan='LIDAR_TOP', nsweeps=self.nsweeps
        )
        lidar_base_features = lidar_pc.points[:4, :]
        lidar_temporal = np.vstack([lidar_base_features, lidar_times]).T
        
        num_lidar = lidar_temporal.shape[0]
        if num_lidar >= self.max_lidar_points:
            indices = np.random.choice(num_lidar, self.max_lidar_points, replace=False)
        else:
            indices = np.random.choice(num_lidar, self.max_lidar_points, replace=True)
        lidar_tensor = torch.tensor(lidar_temporal[indices, :], dtype=torch.float32)
        
        # 4. Ground Truth Mask
        scene = self.nusc.get('scene', sample['scene_token'])
        location = self.nusc.get('log', scene['log_token'])['location']
        ego_pose = self.nusc.get('ego_pose', cam_data['ego_pose_token'])
        
        patch_box = (ego_pose['translation'][0], ego_pose['translation'][1], 40, 40)
        patch_angle = Quaternion(ego_pose['rotation']).yaw_pitch_roll[0]
        
        mask = self.nusc_maps[location].get_map_mask(
            patch_box=patch_box, patch_angle=patch_angle, layer_names=['drivable_area'], canvas_size=(200, 200)
        )
        mask_tensor = torch.tensor(mask[0], dtype=torch.float32).unsqueeze(0)
        
        return {
            'image': img_tensor,
            'radar': radar_tensor,
            'lidar': lidar_tensor,
            'label_mask': mask_tensor
        }

In [5]:
import torch
import torch.nn as nn
import torchvision.models as models

class PointNetExtractor(nn.Module):
    """Extracts features from an unordered point cloud (N points, C channels)"""
    def __init__(self, in_channels, out_features):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, out_features, kernel_size=1),
            nn.BatchNorm1d(out_features),
            nn.ReLU()
        )
        
    def forward(self, x):
        x = x.transpose(1, 2) 
        x = self.mlp(x)
        x = torch.max(x, dim=2, keepdim=False)[0] 
        return x

class BEVFusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        # 1. Image Backbone 
        mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        self.image_backbone = nn.Sequential(*list(mobilenet.features.children()))
        self.image_pool = nn.AdaptiveAvgPool2d((1, 1)) 
        
        # 2. Temporal Point Cloud Backbones
        self.lidar_net = PointNetExtractor(in_channels=5, out_features=128)
        self.radar_net = PointNetExtractor(in_channels=6, out_features=64)
        
        # 3. Fusion Layer
        self.fc = nn.Linear(1472, 256 * 5 * 5)
        
        # 4. BEV Decoder 
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),  
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),   
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),    
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=5, stride=5, padding=0),    
            nn.Conv2d(16, 1, kernel_size=3, padding=1)
        )

    def forward(self, image, lidar, radar):
        img_feat = self.image_pool(self.image_backbone(image)).flatten(1)
        lidar_feat = self.lidar_net(lidar)
        radar_feat = self.radar_net(radar)
        fused = torch.cat([img_feat, lidar_feat, radar_feat], dim=1)
        x = self.fc(fused).view(-1, 256, 5, 5)
        out = self.decoder(x)
        return out

In [6]:
import torch.optim as optim
from torch.amp import autocast, GradScaler
import os

# 1. Hardware Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache() 
torch.backends.cudnn.benchmark = True 

# 2. Initialize the Updated Dataset & DataLoader
print("Loading Fixed-Dimension Dataset...")
temporal_dataset = NuScenesTemporalDataset(nusc, nusc.scene[:10], nsweeps=3)

# UPDATED: We can now safely set batch_size=2
train_loader = DataLoader(temporal_dataset, batch_size=2, shuffle=True, num_workers=0)

# 3. Model Setup
model = BEVFusionModel().to(device)
criterion = torch.nn.BCEWithLogitsLoss() 
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scaler = GradScaler('cuda')

epochs = 5 
save_dir = "./temporal_saved_models_bs2"
os.makedirs(save_dir, exist_ok=True)

print(f"🚀 Starting Temporal Training (Batch Size: 2)...")
print("-" * 45)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for batch_idx, batch in enumerate(train_loader):
        images = batch['image'].to(device)
        lidar = batch['lidar'].to(device)
        radar = batch['radar'].to(device)
        targets = batch['label_mask'].to(device)
        
        optimizer.zero_grad()
        
        with autocast('cuda'):
            predictions = model(images, lidar, radar)
            loss = criterion(predictions, targets)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        
        # Print progress every 10 batches (since batches hold twice the data now)
        if (batch_idx + 1) % 10 == 0:
            avg_loss = running_loss / 10
            print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx+1}/{len(train_loader)}] | Avg BCE Loss: {avg_loss:.4f}")
            running_loss = 0.0 
            
    checkpoint_path = os.path.join(save_dir, f"temporal_bev_epoch_{epoch+1}_bs2.pth")
    torch.save(model.state_dict(), checkpoint_path)
    print(f"💾 Epoch {epoch+1} complete. Model saved to: {checkpoint_path}")
    print("-" * 45)

print("✅ Training with Batch Size 2 Complete!")

Loading Fixed-Dimension Dataset...
🚀 Starting Temporal Training (Batch Size: 2)...
---------------------------------------------
Epoch [1/5] | Batch [10/202] | Avg BCE Loss: 0.6911
Epoch [1/5] | Batch [20/202] | Avg BCE Loss: 0.6902
Epoch [1/5] | Batch [30/202] | Avg BCE Loss: 0.6884
Epoch [1/5] | Batch [40/202] | Avg BCE Loss: 0.6616
Epoch [1/5] | Batch [50/202] | Avg BCE Loss: 0.6793
Epoch [1/5] | Batch [60/202] | Avg BCE Loss: 0.6601
Epoch [1/5] | Batch [70/202] | Avg BCE Loss: 0.6648
Epoch [1/5] | Batch [80/202] | Avg BCE Loss: 0.6500
Epoch [1/5] | Batch [90/202] | Avg BCE Loss: 0.6666
Epoch [1/5] | Batch [100/202] | Avg BCE Loss: 0.6539
Epoch [1/5] | Batch [110/202] | Avg BCE Loss: 0.6409
Epoch [1/5] | Batch [120/202] | Avg BCE Loss: 0.6428
Epoch [1/5] | Batch [130/202] | Avg BCE Loss: 0.6429
Epoch [1/5] | Batch [140/202] | Avg BCE Loss: 0.6354
Epoch [1/5] | Batch [150/202] | Avg BCE Loss: 0.6319
Epoch [1/5] | Batch [160/202] | Avg BCE Loss: 0.6331
Epoch [1/5] | Batch [170/202] | 

In [7]:
import torch
import numpy as np

# 1. Hardware and Model Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BEVFusionModel().to(device)

# 2. Load the trained weights from the BS=2 run
best_model_path = "./temporal_saved_models_bs2/temporal_bev_epoch_5_bs2.pth"
model.load_state_dict(torch.load(best_model_path))
model.eval() 
print(f"✅ Loaded weights from: {best_model_path}")

# 3. Create a Validation DataLoader (Using Scenes 8 and 9)
val_dataset = NuScenesTemporalDataset(nusc, nusc.scene[8:10], nsweeps=3)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)

# 4. IoU Metric Function
def calculate_iou(pred_mask, true_mask, threshold=0.5):
    """Calculates the Intersection over Union for binary masks."""
    pred_bin = (pred_mask > threshold).astype(bool)
    true_bin = true_mask.astype(bool)
    
    intersection = np.logical_and(pred_bin, true_bin).sum()
    union = np.logical_or(pred_bin, true_bin).sum()
    
    if union == 0:
        return 0.0 
    return intersection / union

# 5. Evaluation Loop
total_iou = 0.0
num_batches = len(val_loader)

print(f"Starting Evaluation on {num_batches} batches...")
print("-" * 45)

with torch.no_grad(): 
    for batch_idx, batch in enumerate(val_loader):
        images = batch['image'].to(device)
        lidar = batch['lidar'].to(device)
        radar = batch['radar'].to(device)
        targets = batch['label_mask']
        
        with torch.amp.autocast('cuda'):
            raw_logits = model(images, lidar, radar)
            probabilities = torch.sigmoid(raw_logits)
            
        pred_numpy = probabilities.squeeze().cpu().numpy()
        target_numpy = targets.squeeze().cpu().numpy()
        
        batch_iou = calculate_iou(pred_numpy, target_numpy)
        total_iou += batch_iou
        
        if (batch_idx + 1) % 10 == 0:
            print(f"Evaluated Batch [{batch_idx+1}/{num_batches}] | Current Batch IoU: {batch_iou:.4f}")

# 6. Final Results
mean_iou = total_iou / num_batches
print("-" * 45)
print(f"🏆 Final Mean IoU (Batch Size 2): {mean_iou * 100:.2f}%")

✅ Loaded weights from: ./temporal_saved_models_bs2/temporal_bev_epoch_5_bs2.pth
Starting Evaluation on 80 batches...
---------------------------------------------
Evaluated Batch [10/80] | Current Batch IoU: 0.7477
Evaluated Batch [20/80] | Current Batch IoU: 0.7630
Evaluated Batch [30/80] | Current Batch IoU: 0.7478
Evaluated Batch [40/80] | Current Batch IoU: 0.7307
Evaluated Batch [50/80] | Current Batch IoU: 0.9026
Evaluated Batch [60/80] | Current Batch IoU: 0.9045
Evaluated Batch [70/80] | Current Batch IoU: 0.8920
Evaluated Batch [80/80] | Current Batch IoU: 0.8929
---------------------------------------------
🏆 Final Mean IoU (Batch Size 2): 82.05%


In [8]:
import cv2
import torch
import numpy as np
import os

# 1. Set model to evaluation mode
model.eval()

# 2. Load a continuous sequence (Scene 8) for the video
print("Loading continuous scene for video generation...")
video_dataset = NuScenesTemporalDataset(nusc, nusc.scene[8:9], nsweeps=3)
# Keep batch size 1 and shuffle False to maintain temporal order!
video_loader = DataLoader(video_dataset, batch_size=1, shuffle=False, num_workers=0)

# 3. Video Writer Configuration
# Dimensions: Camera (400x224) + Pred (224x224) + GT (224x224)
frame_width = 400 + 224 + 224 
frame_height = 224
fps = 2.0 # nuScenes samples at 2Hz

out_path = 'bev_driving_inference.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video = cv2.VideoWriter(out_path, fourcc, fps, (frame_width, frame_height))

print(f"🎥 Rendering frames to {out_path}...")
print("-" * 45)

with torch.no_grad():
    for batch_idx, batch in enumerate(video_loader):
        # A. Forward Pass
        images = batch['image'].to(device)
        lidar = batch['lidar'].to(device)
        radar = batch['radar'].to(device)
        targets = batch['label_mask']
        
        with torch.amp.autocast('cuda'):
            raw_logits = model(images, lidar, radar)
            probabilities = torch.sigmoid(raw_logits)
            
        # B. Process Camera Image
        img_vis = images.squeeze().cpu().permute(1, 2, 0).numpy()
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_vis = np.clip((img_vis * std) + mean, 0, 1)
        img_vis = (img_vis * 255).astype(np.uint8)
        img_vis = cv2.cvtColor(img_vis, cv2.COLOR_RGB2BGR) # Convert to OpenCV BGR
        
        # C. Process Prediction Mask (Heatmap)
        pred_mask = probabilities.squeeze().cpu().numpy()
        pred_mask_vis = (pred_mask * 255).astype(np.uint8)
        pred_mask_color = cv2.applyColorMap(pred_mask_vis, cv2.COLORMAP_MAGMA)
        pred_mask_color = cv2.resize(pred_mask_color, (224, 224)) 
        
        # D. Process Ground Truth Mask
        true_mask = targets.squeeze().cpu().numpy()
        true_mask_vis = (true_mask * 255).astype(np.uint8)
        true_mask_color = cv2.cvtColor(true_mask_vis, cv2.COLOR_GRAY2BGR)
        true_mask_color = cv2.resize(true_mask_color, (224, 224))
        
        # E. Add Text Labels
        cv2.putText(img_vis, "Input Camera", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv2.putText(pred_mask_color, "Predicted Free Space", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        cv2.putText(true_mask_color, "Ground Truth", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        
        # F. Stitch Together & Write
        composite_frame = np.hstack((img_vis, pred_mask_color, true_mask_color))
        video.write(composite_frame)
        
        if (batch_idx + 1) % 10 == 0:
            print(f"Rendered Frame [{batch_idx+1}/{len(video_loader)}]")

video.release()
print("-" * 45)
print(f"✅ Video successfully saved as: {out_path}")

Loading continuous scene for video generation...
🎥 Rendering frames to bev_driving_inference.mp4...
---------------------------------------------
Rendered Frame [10/40]
Rendered Frame [20/40]
Rendered Frame [30/40]
Rendered Frame [40/40]
---------------------------------------------
✅ Video successfully saved as: bev_driving_inference.mp4


In [9]:
import cv2
import torch
import numpy as np
from PIL import Image

model.eval()

# 1. Pre-compute the 2Hz BEV predictions for Scene 8 
print("Pre-computing 2Hz BEV predictions to keep VRAM clean...")
scene = nusc.scene[8]
val_dataset = NuScenesTemporalDataset(nusc, [scene], nsweeps=3)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)

# Dictionary to store predictions, keyed by the sample_token
bev_cache = {}

with torch.no_grad():
    for i, batch in enumerate(val_loader):
        images = batch['image'].to(device)
        lidar = batch['lidar'].to(device)
        radar = batch['radar'].to(device)
        
        with torch.amp.autocast('cuda'):
            raw_logits = model(images, lidar, radar)
            prob = torch.sigmoid(raw_logits).squeeze().cpu().numpy()
        
        # Format Prediction
        pred_vis = (prob * 255).astype(np.uint8)
        pred_color = cv2.applyColorMap(pred_vis, cv2.COLORMAP_MAGMA)
        pred_color = cv2.resize(pred_color, (224, 224))
        
        # Format Ground Truth
        gt_mask = batch['label_mask'].squeeze().cpu().numpy()
        gt_vis = (gt_mask * 255).astype(np.uint8)
        gt_color = cv2.cvtColor(gt_vis, cv2.COLOR_GRAY2BGR)
        gt_color = cv2.resize(gt_color, (224, 224))
        
        # Save to cache using the exact token
        sample_token = val_dataset.samples[i]['token']
        bev_cache[sample_token] = (pred_color, gt_color)

# 2. Traverse the raw 12Hz Camera Linked List
print("Extracting 12Hz intermediate camera sweeps from the dataset...")
first_cam_token = nusc.get('sample', scene['first_sample_token'])['data']['CAM_FRONT']
current_cam_token = first_cam_token

out_path = 'bev_driving_12hz_extrapolated.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# Native nuScenes camera framerate is exactly 12.0 FPS
video = cv2.VideoWriter(out_path, fourcc, 12.0, (400 + 224 + 224, 224)) 

# Initialize blank arrays to hold the asynchronous maps
last_pred = np.zeros((224, 224, 3), dtype=np.uint8)
last_gt = np.zeros((224, 224, 3), dtype=np.uint8)
frame_count = 0

while current_cam_token != '':
    cam_data = nusc.get('sample_data', current_cam_token)
    
    # Extract the raw, high-framerate image
    img_path = nusc.get_sample_data_path(current_cam_token)
    img = Image.open(img_path).convert('RGB')
    img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    img_cv = cv2.resize(img_cv, (400, 224))
    
    # ASYNCHRONOUS UPDATE: If this frame is an annotated keyframe, update the BEV maps
    if cam_data['is_key_frame']:
        sample_token = cam_data['sample_token']
        if sample_token in bev_cache:
            last_pred, last_gt = bev_cache[sample_token]
            
    # Draw Labels
    cv2.putText(img_cv, "Input Camera (12Hz)", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    cv2.putText(last_pred, "Pred BEV (2Hz Async)", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    cv2.putText(last_gt, "GT Mask (2Hz Async)", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    # Stitch and Write
    composite = np.hstack((img_cv, last_pred, last_gt))
    video.write(composite)
    
    frame_count += 1
    if frame_count % 30 == 0:
         print(f"Extracted {frame_count} frames...")
    
    # Move via linked list pointer to the next 12Hz frame
    current_cam_token = cam_data['next']

video.release()
print("-" * 45)
print(f"✅ Extrapolated 12fps Video saved as: {out_path}")

Pre-computing 2Hz BEV predictions to keep VRAM clean...
Extracting 12Hz intermediate camera sweeps from the dataset...
Extracted 30 frames...
Extracted 60 frames...
Extracted 90 frames...
Extracted 120 frames...
Extracted 150 frames...
Extracted 180 frames...
Extracted 210 frames...
---------------------------------------------
✅ Extrapolated 12fps Video saved as: bev_driving_12hz_extrapolated.mp4
